In [ ]:
import tensorflow as tf
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    for device in physical_devices:
        tf.config.experimental.set_memory_growth(device, True)
        print('{} memory growth: {}'.format(device, tf.config.experimental.get_memory_growth(device)))
else:
    print("Not enough GPU hardware devices available")

In [ ]:
from tensorflow import keras
from keras.models import Sequential
from keras.layers.convolutional import MaxPooling2D
from keras.layers import Activation, Conv2D, Flatten, Dense,Dropout, BatchNormalization
from keras.optimizers import adam_v2
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd  
import seaborn as sn
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, classification_report
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import StratifiedKFold
import pickle

In [ ]:
#If necessary, download the augmented image dataset from Zenodo and place it into the train folder.

source_name="tissue_augmentate"
model_name="tissue"

# source_name="culture_augmentate"
# model_name="culture"

# source_name="neuronmixture_augmentate"
# model_name="neuronmixture"

In [ ]:
folder = os.listdir("../data/train/"+source_name)

In [ ]:
os.makedirs("../figure/"+model_name+"/extest", exist_ok = True)
os.makedirs("../result_dir", exist_ok = True)
os.makedirs("../result_dir/"+model_name, exist_ok = True)

In [ ]:
def cmx_figure_annot(y_true, y_pred, save_path, save_name):
    
    labels = sorted(list(set(y_true)|set(y_pred)))
    cmx_data = confusion_matrix(y_true, y_pred, labels=labels)
    
    df_cmx = pd.DataFrame(cmx_data, index=true_label, columns=pred_label)
 
    fig=plt.figure(figsize = (10,7))
    sn.heatmap(df_cmx, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("predict_classes")
    plt.ylabel("true_classes")
    fig.savefig(save_path+"/"+save_name+str(rd)+"_annot.tiff", dpi=600)  
    plt.show()

In [ ]:
def cmx_figure(y_true, y_pred, save_path, save_name):
    
    labels = sorted(list(set(y_true)|set(y_pred)))
    cmx_data = confusion_matrix(y_true, y_pred, labels=labels)
    
    df_cmx = pd.DataFrame(cmx_data, index=true_label, columns=pred_label)
 
    fig=plt.figure(figsize = (10,7))
    sn.heatmap(df_cmx, fmt="d", cmap="Blues", xticklabels = False, yticklabels = False)
    plt.xlabel("predict_classes")
    plt.ylabel("true_classes")
    fig.savefig(save_path+"/"+save_name+str(rd)+".tiff", dpi=600) 
    plt.show()

In [ ]:
dense_size  = len(folder)
classes_number = len(folder)

In [ ]:
#k-fold
fold_number = 8

batch_size = 128
image_size = 150

epochs = 100

seed = 0
np.random.seed(seed)

In [ ]:
#savepath for cmx
true_label = os.listdir("../data/train/"+source_name)
pred_label = os.listdir("../data/train/"+source_name)
save_path = "../figure/"+model_name
save_name = "cmx"

In [ ]:
X_train=np.load(file="../data/train/Numpy/"+model_name+"/"+model_name+"_X_train.npy")
y_train=np.load(file="../data/train/Numpy/"+model_name+"/"+model_name+"_y_train.npy")
X_val=np.load(file="../data/train/Numpy/"+model_name+"/"+model_name+"_X_val.npy")
y_val=np.load(file="../data/train/Numpy/"+model_name+"/"+model_name+"_y_val.npy")

In [ ]:
X_dev = np.concatenate([X_train, X_val], 0)
y_dev = np.concatenate([y_train, y_val], 0)

In [ ]:
SKFold = StratifiedKFold(n_splits=fold_number, shuffle=True, random_state=seed)
train_acc=[]
train_loss=[]
train_acc_history=[]
train_loss_history=[]
cvscores = []
cvloss=[]
cvscores_history=[]
cvloss_history=[]
y=[]
y_score=[]
rd=1

for train, test in SKFold.split(X_dev, y_dev):
    model = Sequential()
    model.add(Conv2D(64, (3, 3), activation="relu", padding="same",input_shape=(image_size,image_size,1)))
    model.add(BatchNormalization())
    model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.2))
    
    model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.2))

    model.add(Conv2D(64, (3, 3),activation="relu", padding="same"))
    model.add(Conv2D(64, (3, 3),activation="relu", padding="same"))
    model.add(BatchNormalization())
    model.add(Conv2D(64, (3, 3),activation="relu", padding="same"))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.2))

    model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.3))
    
    model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.3))

    model.add(Flatten())
    model.add(Dense(512, activation="relu"))
    model.add(Dropout(0.4))
    model.add(Dense(256, activation="relu"))
    model.add(Dropout(0.3))
    model.add(Dense(128, activation="relu"))
    model.add(Dropout(0.2))
    model.add(Dense(dense_size, activation="softmax"))

    model.compile(loss="categorical_crossentropy", optimizer=adam_v2.Adam(learning_rate=1e-3, decay=1e-3), metrics=["accuracy"])
    result=model.fit(X_dev[train], keras.utils.to_categorical(y_dev[train], classes_number),
                    batch_size=batch_size,
                    epochs=epochs,
                    verbose=1,
                    validation_data=(X_dev[test], keras.utils.to_categorical(y_dev[test], classes_number))
                    )
    
    #accuracy graph
    fig=plt.figure()
    plt.plot(range(1, epochs+1), result.history["accuracy"], label="training")
    plt.plot(range(1, epochs+1), result.history["val_accuracy"], label="validation")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.legend()
    fig.savefig("../figure/"+model_name+"/accuracy"+str(rd)+".tiff", dpi=600)
    plt.show()

    #loss graph
    fig=plt.figure()
    plt.plot(range(1, epochs+1), result.history["loss"], label="training")
    plt.plot(range(1, epochs+1), result.history["val_loss"], label="validation")
    plt.xlabel("Epochs")
    plt.ylabel("loss")
    plt.legend()
    fig.savefig("../figure/"+model_name+"/loss"+str(rd)+".tiff", dpi=600)
    plt.show()
    
    #save history
    train_acc_history.append(result.history["accuracy"])
    train_loss_history.append(result.history["loss"])
    cvscores_history.append(result.history["val_accuracy"])
    cvloss_history.append(result.history["val_loss"])
    
    #cmx
    y_pred = np.argmax(model.predict(X_dev[test]), axis=1)
    y_true = np.argmax(keras.utils.to_categorical(y_dev[test], classes_number), axis=1) 
    print(confusion_matrix(y_true, y_pred))
    cmx_figure_annot(y_true, y_pred, save_path, save_name)
    cmx_figure(y_true, y_pred, save_path, save_name)
    
    #report
    labels = sorted(list(set(y_true)|set(y_pred)))
    cmx_data= confusion_matrix(y_true, y_pred, labels=labels)
    precision_recall_fscore_support(y_true, y_pred, average="macro")
    print(classification_report(y_true, y_pred, digits=6))
    
    #ROC
    y_binarize = label_binarize(y_dev[test], classes=[0, 1, 2, 3])    
    y_prob = model.predict(X_dev[test])
    y.append(y_binarize)
    y_score.append(y_prob)
    
    #save weight 
    print("save json")
    json_string = model.to_json()
    f=open("../result_dir/"+model_name+"/"+model_name+"_model"+str(rd)+".json", "w")
    f.write(json_string)
    f.close()
    print("save weights")
    model.save_weights("../result_dir/"+model_name+"/"+model_name+"_model"+str(rd)+"_weights.hdf5")
    rd+=1
    
    #training score
    print("{}: {:.4%}".format("train_acc", result.history["accuracy"][-1]))
    print("{}: {:.6g}".format("train_loss", result.history["loss"][-1]))
    train_acc.append(result.history["accuracy"][-1]*100)
    train_loss.append(result.history["loss"][-1])

    #validation score
    scores = model.evaluate(X_dev[test], keras.utils.to_categorical(y_dev[test], classes_number), verbose=0)
    print("{}: {:.4%}".format(model.metrics_names[1], scores[1]))
    print("{}: {:.6g}".format(model.metrics_names[0], scores[0]))
    cvscores.append(scores[1] * 100)
    cvloss.append(scores[0])

#training & validation score mean
print("train_acc: {:.6g} (+/-{:.6g}%)".format(np.mean(train_acc), np.std(train_acc)))
print("train_loss: {:.6g} (+/-{:.6g})".format(np.mean(train_loss), np.std(train_loss)))
print("accuracy: {:.6g} (+/-{:.6g}%)".format(np.mean(cvscores), np.std(cvscores)))
print("loss: {:.6g} (+/-{:.6g})".format(np.mean(cvloss), np.std(cvloss)))

In [ ]:
with open("../result_dir/"+model_name+"/"+model_name+"_y.pickle", mode="wb") as f:
    pickle.dump(y, f)
with open("../result_dir/"+model_name+"/"+model_name+"_y_score.pickle", mode="wb") as f:
    pickle.dump(y_score, f)
with open("../result_dir/"+model_name+"/"+model_name+"_train_acc.pickle", mode="wb") as f:
    pickle.dump(train_acc, f)
with open("../result_dir/"+model_name+"/"+model_name+"_train_loss.pickle", mode="wb") as f:
    pickle.dump(train_loss, f)
with open("../result_dir/"+model_name+"/"+model_name+"_train_acc_history.pickle", mode="wb") as f:
    pickle.dump(train_acc_history, f)
with open("../result_dir/"+model_name+"/"+model_name+"_train_loss_history.pickle", mode="wb") as f:
    pickle.dump(train_loss_history, f)
with open("../result_dir/"+model_name+"/"+model_name+"_cvscores.pickle", mode="wb") as f:
    pickle.dump(cvscores, f)
with open("../result_dir/"+model_name+"/"+model_name+"_cvloss.pickle", mode="wb") as f:
    pickle.dump(cvloss, f)
with open("../result_dir/"+model_name+"/"+model_name+"_cvscores_history.pickle", mode="wb") as f:
    pickle.dump(cvscores_history, f)
with open("../result_dir/"+model_name+"/"+model_name+"_cvloss_history.pickle", mode="wb") as f:
    pickle.dump(cvloss_history, f)